# PRL: фрагментация и коагуляция на одной фигуре

Две панели, одна под другой, **общая ось масс**.

Обе коробки живут на одном и том же диапазоне, `m` от 1 до 1e6, но поток идёт
навстречу:

| | инжекция | сток | поток |
|---|---|---|---|
| **фрагментация** | 1e6 | 1 | справа налево |
| **коагуляция** | 1 | 1e6 | слева направо |

Поэтому `sharex` тут не косметика: это и есть содержание рисунка.

Панели — ровно те PRL-блоки, что стоят в конце двух тетрадей анализа, без
переделок:

* **сверху** — `Analysis_open_fragmentation_conservative_geometric`, последняя
  ячейка: `m^2 dN/dm` и поколения, окрашенные по `tau(g)`;
* **снизу** — `Analysis_open_coagulation_conservative_geometric`,
  `PRL_STYLE_COMPENSATED_SPECTRUM`: `m^2 dN/dm` и изохроны, окрашенные по
  возрасту, с логнормальным фитом.

Подписи панелей стоят **внутри рамки**, по центру сверху, а не заголовками.

Ничего не считается заново и ничего не пишется на диск: тетрадь только читает
два `.npz` и рисует.


## Загрузка обоих прогонов

In [ ]:
import os, glob, importlib, numpy as np, matplotlib.pyplot as plt

import BF_analysis as AN
#  RELOAD, всегда.  Jupyter кэширует модуль между запусками, и правка
#  BF_analysis.py без перезапуска ядра молча оставляет СТАРЫЙ оценщик --
#  он не падает, он отвечает иначе.
importlib.reload(AN)

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.figsize": (9, 3.2)})
print("движок:", AN.BF.__name__, "| слой измерения:", AN.__file__)

# ---- какие файлы брать.  Единственное, что тут стоит трогать. ----------
PAT_FRAG = "runs/open_fragmentation_conservative_geometric_f*.npz"
PAT_COAG = "runs/open_coagulation_conservative_geometric*.npz"
FILE_FRAG = None      # <<-- явное имя, тогда шаблон не нужен
FILE_COAG = None


def _newest(pattern, explicit, what):
    """Самый свежий файл под шаблон.  Молча брать всё, что подцепил glob,
    нельзя: ручные копии с датой в имени -- это другие прогоны, и половина
    величин у старых просто отсутствует."""
    if explicit is not None:
        p = explicit if ("/" in explicit or os.sep in explicit) else os.path.join("runs", explicit)
        if not os.path.exists(p):
            raise FileNotFoundError("%s: нет файла %s" % (what, p))
        return p
    found = sorted(glob.glob(pattern), key=os.path.getmtime)
    if not found:
        raise FileNotFoundError(
            "%s: ничего не подходит под %s -- прогони соседнюю run-тетрадь" % (what, pattern))
    if len(found) > 1:
        print("%s: найдено %d файлов, беру последний по времени; пропущены:"
              % (what, len(found)))
        for f in found[:-1]:
            print("   пропущен  %s" % os.path.basename(f))
    return found[-1]


FN_FRAG = _newest(PAT_FRAG, FILE_FRAG, "фрагментация")
FN_COAG = _newest(PAT_COAG, FILE_COAG, "коагуляция")

R_FRAG = AN.load(FN_FRAG);  S_FRAG = AN.spectrum(R_FRAG)
R_COAG = AN.load(FN_COAG);  S_COAG = AN.spectrum(R_COAG)

#  Изохроны коагуляции выбираются по плато закона роста -- то же окно, что
#  и в родной тетрадке, чтобы кривые были те же самые.
GROW_COAG = AN.growth_compare(R_COAG) if "iso_counts" in R_COAG else None

# ---- что должно быть в файлах, проверяем СРАЗУ ------------------------
for need, r, fn, what in ((("gen_counts", "gen_tau", "gen_snapshots"), R_FRAG, FN_FRAG, "фрагментация"),
                          (("iso_counts", "iso_dndm", "iso_snapshots"), R_COAG, FN_COAG, "коагуляция")):
    miss = [k for k in need if k not in r]
    if miss:
        raise KeyError("%s (%s): в прогоне нет %s -- движок старше нужного, "
                       "или загрузилось не то. Проверь шаблон выше."
                       % (what, os.path.basename(fn), ", ".join(miss)))


def _scales(r):
    """m_inj и m_sink, откуда бы движок их ни записал."""
    a = r["meta"].get("analysis", {}) or {}
    mi = a.get("m_inj", r["meta"].get("injection_mass"))
    ms = a.get("m_sink", r["meta"].get("sink_mass"))
    return float(mi), float(ms)


for what, r, fn in (("фрагментация", R_FRAG, FN_FRAG), ("коагуляция", R_COAG, FN_COAG)):
    mi, ms = _scales(r)
    pl = AN.spectrum_plateau(r, spec=(S_FRAG if r is R_FRAG else S_COAG))
    print("%-14s %-52s stop = %s" % (what, os.path.basename(fn), r["meta"].get("stop_reason")))
    print("               инжекция %.3g -> сток %.3g | снимков %d | alpha_плато = %+.3f +- %.3f"
          % (mi, ms, np.asarray(r["dndm"]).shape[0], pl["alpha"], pl.get("scatter", np.nan)))


## Фигура

Всё, что стоит трогать, собрано в первом блоке ячейки. Дальше — два блока,
скопированных из родных тетрадей, и раскладка.

In [ ]:
# ======================================================================
#  PRL, две панели на общей оси масс:
#     сверху  ФРАГМЕНТАЦИЯ  -- спектр и ПОКОЛЕНИЯ, цвет по tau(g)
#     снизу   КОАГУЛЯЦИЯ    -- спектр и ИЗОХРОНЫ,  цвет по возрасту
# ======================================================================
#  На обеих панелях нарисовано одно и то же и без нормировок:
#     спектр   -> dN/dm, умножить на m^2
#     кусок    -> (dN/dm) этого куска, умножить на m^2
#  Куски (поколения сверху, изохроны снизу) -- непересекающееся разбиение
#  популяции, поэтому каждый лежит НИЖЕ спектра, а их сумма его
#  восстанавливает.  Спектр нарисован без моделей: точки с ошибками.
#  Показатель стоит подписью, как число, а не как проведённая кривая.
#
#  Единственная модель на всей фигуре -- логнормаль на нижней панели, и она
#  подписана как фит.

# ---- ЧТО РИСОВАТЬ ----------------------------------------------------
GEN_PICK   = (4, 8, 14, 18, 24)  # <<-- поколения, верхняя панель
GEN_REBIN  = 3                   # слить по стольку бинов (0.1 dex -> 0.3 dex)
MIN_CNT    = 30.0                # порог по СЫРЫМ отсчётам на бин
N_CURVES   = 8                   # <<-- целевое число изохрон, нижняя панель
AGE_UNIT   = "t_c"               # "t_c" -- возраст в СОБСТВЕННЫХ временах
                                 #          столкновения каждого прогона; тогда
                                 #          шкала ОДНА на обе панели.
                                 # число (напр. 1e-6) -- абсолютное время, и
                                 #          тогда шкал две, по одной на панель.

SHOW_HONEST    = True            # серая линия сверху: сумма ВСЕХ поколений
SHOW_LOGNORMAL = True            # чёрная кривая снизу: логнормальный фит
LOGNORM_LABEL   = "lognormal"    # "" -- не подписывать
LOGNORM_LABEL_X = 0.80           # по горизонтали, в долях рамки
LOGNORM_LABEL_Y = 0.72           # высота подписи логнормали, в долях рамки
N_YTICKS       = 4               # десятичных тиков по y на панель

SHOW_GAUSS     = False            # чёрная кривая СВЕРХУ: гауссиана по одному
                                 # поколению -- пара к логнормали внизу
GEN_FIT        = 2               # <<-- КАКОЕ из показанных поколений фитировать.
                                 # Это ИНДЕКС в GEN_PICK, а не сам g:
                                 # 0 = первое показанное, 2 = ТРЕТЬЕ.
                                 # None = не фитировать.
GAUSS_FIT_LOG  = True            # фитить в ЛОГАРИФМЕ y.  См. комментарий в
                                 # панели: в линейном y фит уезжает.
GAUSS_GUARD    = (0.0, 0.0)      # сколько ДЕКАД отрезать от стока и от инжекции
                                 # перед фитом.  (0, 0) = фитить ровно то, что
                                 # нарисовано.  Систематика окна печатается
                                 # всегда, независимо от этого выбора.
GAUSS_LABEL    = "gaussian"      # "" -- не подписывать
GAUSS_LABEL_X  = 0.55            # по горизонтали, в долях рамки
GAUSS_LABEL_Y  = 0.58            # по вертикали, в долях рамки

LABELS   = ("Fragmentation", "Coagulation")   # <<-- текст ВНУТРИ рамки
LABEL_Y  = 0.94                  # по вертикали, в долях рамки
LABEL_X  = 0.5                   # по горизонтали

XLIM      = None                 # None = объединение обоих прогонов
YLIM_FRAG = (1e4, 3e8)                 # None = авто
YLIM_COAG = (1e4, 3e7)           # None = авто
FIGSIZE   = (3.375, 2.85)        # СТАНДАРТНАЯ одноколоночная PRL.
                                 # 3.375 in = 8.6 cm -- ширина колонки PRL.
CBAR_PAD  = 0.03                 # поджать цветовую шкалу сверху и снизу

SAVEFIG   = None                 # напр. "fig_prl_frag_coag.pdf".
                                 # None = НИЧЕГО не писать на диск.

from matplotlib.colors import LogNorm

CMAP = plt.cm.viridis

prl_rc = {
    "figure.dpi": 160, "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "legend.fontsize": 6.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.linewidth": 0.8, "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True, "axes.grid": False,
}


def _age_scale(r):
    """Чем делить возраст и как подписать шкалу.

    AGE_UNIT = "t_c" -- делим на СОБСТВЕННОЕ время столкновения прогона.  Это
    единственный способ посадить обе панели на одну цветовую шкалу честно: у
    фрагментации t_c = 1.25e-7, у коагуляции 2.5e-7, и абсолютные возрасты в
    один и тот же цвет красить нельзя -- они в разных единицах.
    Число -- абсолютное время, тогда шкал две."""
    if AGE_UNIT == "t_c":
        a = r["meta"].get("analysis", {}) or {}
        tc = a.get("t_c")
        if tc is None or not np.isfinite(float(tc)) or float(tc) <= 0:
            return None, None            # нет t_c -- откат на абсолютную шкалу
        return float(tc), r"$\tau/t_c$"
    u = float(AGE_UNIT)
    return u, r"$\tau$  ($10^{%d}$)" % round(np.log10(u))


def _rebin(F, edges, k):
    """Слить по k бинов, сохранив ПЛОТНОСТЬ: суммируем счёт, делим на новую
    ширину.  Возвращает (центры, плотность, СЫРЫЕ отсчёты) -- отсчёты нужны
    для порога."""
    n = (edges.size - 1) // k * k
    w = np.diff(edges)[:n]
    cnt = (np.asarray(F)[:n] * w).reshape(-1, k).sum(1)
    e2 = np.concatenate([edges[:n:k], [edges[n]]])
    return np.sqrt(e2[:-1] * e2[1:]), cnt / np.diff(e2), cnt


def _age_ticks(cb, age, n_max=4, label=None):
    """Тики -- сами возрасты нарисованных кривых.  Соседние прореживаются: на
    больших g/tau возрасты сходятся и подписи садятся друг на друга.  Последний
    обязателен -- он задаёт верх шкалы; если он сел вплотную к предыдущему,
    выбрасывается ПРЕДЫДУЩИЙ, а не он."""
    age = np.asarray(age, float)
    la = np.log10(age)
    span = float(la.max() - la.min())
    gap = max(0.06 * span, span / max(n_max - 1, 1)) if span > 0 else 0.0
    keep = [0]
    for i in range(1, la.size):
        if la[i] - la[keep[-1]] > gap - 1e-12:
            keep.append(i)
    last = la.size - 1
    if keep[-1] != last:
        if len(keep) > 1 and (la[last] - la[keep[-1]]) < 0.6 * gap:
            keep[-1] = last
        else:
            keep.append(last)
    cb.set_ticks(age[keep])
    cb.set_ticklabels(["%.2f" % a for a in age[keep]])
    cb.minorticks_off()
    cb.set_label(label if label is not None else r"$\tau$", labelpad=2, fontsize=7)
    cb.ax.tick_params(direction="in", length=2.0, width=0.7, labelsize=6.0)


def _marks(ax, m_inj, m_sink):
    """Инжекция и сток вертикальными пунктирами.  Подпись отъезжает от края
    коробки внутрь, поэтому сторона зависит от того, что где."""
    for m0, name in ((m_inj, "Injection"), (m_sink, "Sink")):
        ax.axvline(m0, color="0.1", lw=0.8, ls="--", zorder=1)
        inward = 1.35 if m0 < np.sqrt(m_inj * m_sink) else 1 / 1.35
        ax.text(m0 * inward, 0.95, name, transform=ax.get_xaxis_transform(),
                ha=("left" if inward > 1 else "right"), va="top",
                fontsize=6.5, color="0.1")


# ======================================================================
#  ПАНЕЛЬ A -- фрагментация: спектр + поколения
#  (блок из Analysis_open_fragmentation_..., последняя ячейка)
# ======================================================================
def panel_fragmentation(fig, ax, cax):
    r, s = R_FRAG, S_FRAG
    m   = np.asarray(r["centers"], float)
    ed  = np.asarray(r["edges"], float)
    H   = np.asarray(r["gen_counts"], float)   # (g, bin), только честные частицы
    nsn = max(float(r["gen_snapshots"]), 1.0)
    tau_g = np.asarray(r["gen_tau"], float)
    m_inj, m_sink = _scales(r)

    pick = np.array([q for q in GEN_PICK
                     if 0 <= q < H.shape[0] and H[q].sum() > 0 and np.isfinite(tau_g[q])])
    if pick.size == 0:
        raise RuntimeError("ни одно из GEN_PICK не имеет статистики или tau(g): %r" % (GEN_PICK,))
    skipped = [q for q in GEN_PICK if q not in pick]
    if skipped:
        print("фрагментация: пропущены (нет статистики или tau = nan; g = 0 не 'достигается'):",
              skipped)
    age = tau_g[pick] / AGE_DIV_A

    y_all = []

    # ---- спектр: m^2 dN/dm, точки с ошибками, без моделей -------------
    ok_s = np.isfinite(s["F"]) & (s["F"] > 0) & np.isfinite(m) & (m > 0)
    ax.errorbar(m[ok_s], (s["F"] * m**2)[ok_s], yerr=(s["sigma"] * m**2)[ok_s],
                fmt="o", ms=2.0, mfc="white", mec="0.25", mew=0.5,
                ecolor="0.75", elinewidth=0.45, capsize=1.0, color="0.25", zorder=4)
    y_all.append((s["F"] * m**2)[ok_s])

    # ---- сумма всех поколений = честная подвыборка --------------------
    if SHOW_HONEST:
        mh, Fh_raw, ch = _rebin(H.sum(0) / np.diff(ed), ed, GEN_REBIN)
        Fh = Fh_raw / nsn
        okh = np.isfinite(Fh) & (Fh > 0) & (ch >= MIN_CNT)
        ax.loglog(mh[okh], (Fh * mh**2)[okh], "-", lw=1.1, color="0.60", zorder=2)
        y_all.append((Fh * mh**2)[okh])

    # ---- отдельные поколения, цвет по tau(g) --------------------------
    #  Хэндлы кривых сохраняются: если шкала одна на обе панели, цвет
    #  пересчитывается ПОСЛЕ того, как известен общий диапазон.
    global LINES_A
    LINES_A, keep_age, keep_g = [], [], []
    norm = LogNorm(vmin=age.min(), vmax=max(age.max(), age.min() * 1.0001))
    for gg, tk in zip(pick, age):
        mr, Fr_raw, cnt = _rebin(H[gg] / np.diff(ed), ed, GEN_REBIN)
        Fr = Fr_raw / nsn
        ok = np.isfinite(Fr) & (Fr > 0) & (cnt >= MIN_CNT)
        if not ok.any():
            continue
        h, = ax.loglog(mr[ok], (Fr * mr**2)[ok], "-", lw=1.0,
                       color=CMAP(norm(tk)), alpha=0.92, zorder=3)
        LINES_A.append(h); keep_age.append(tk); keep_g.append(gg)
        y_all.append((Fr * mr**2)[ok])
    #  Дальше работаем ТОЛЬКО с тем, что реально нарисовано: поколение из
    #  GEN_PICK могло не пройти порог и вылететь в continue, и тогда индекс
    #  "третье показанное" указывал бы не на то, что видно на картинке.
    age = np.asarray(keep_age, float)
    pick = np.asarray(keep_g, int)

    # ---- ГАУССИАНА по ОДНОМУ поколению --------------------------------
    #  Поколение g -- это сумма g независимых шагов по ln m.  Значит в ln m
    #  оно гауссиана, а в самой m -- логнормаль.  Функция ровно та же самая,
    #  что на нижней панели, lognormal_dndm: фит идёт по СЫРОМУ dN/dm, а на
    #  график кладётся умноженным на m^2 -- как внизу.
    #
    #  Веса моментной оценки -- СЫРЫЕ отсчёты cnt, а не плотность: это и есть
    #  число частиц в бине, то есть настоящий статистический вес.
    gauss_info = None
    if SHOW_GAUSS and GEN_FIT is not None and pick.size and 0 <= GEN_FIT < pick.size:
        g_fit = int(pick[GEN_FIT])
        mr, Fr_raw, cnt = _rebin(H[g_fit] / np.diff(ed), ed, GEN_REBIN)
        Fr = Fr_raw / nsn
        base = np.isfinite(mr) & np.isfinite(Fr) & (mr > 0) & (Fr > 0) & (cnt >= MIN_CNT)

        def _fit(guard_lo, guard_hi):
            """Логнормаль по одному поколению в окне, урезанном на guard декад
            от стока и от инжекции.  Возвращает None, если бинов меньше четырёх."""
            fm = base & (mr > m_sink * 10.0 ** guard_lo) & (mr < m_inj / 10.0 ** guard_hi)
            if fm.sum() < 4:
                return None
            x, y, w = mr[fm], Fr[fm], cnt[fm]
            lx = np.log(x)
            mu0 = float(np.average(lx, weights=w))
            sg0 = max(float(np.sqrt(np.average((lx - mu0) ** 2, weights=w))), 0.15)
            a0 = float(w.sum() / nsn)
            try:
                from scipy.optimize import curve_fit
                if GAUSS_FIT_LOG:
                    #  ФИТ В ЛОГАРИФМЕ y, И ЭТО НЕ ПРИДИРКА.  Поколение живёт на
                    #  пяти декадах, значит dN/dm меняется на много порядков.
                    #  МНК в линейном y видит только самые большие значения -- те,
                    #  что у стока, -- и остальные четыре декады для него шум: на
                    #  этих данных он уводил mu на +17 при ln m_inj = 13.8, то есть
                    #  ставил центр пакета ВЫШЕ массы инжекции.  В логарифме каждая
                    #  декада весит одинаково, и mu садится на моментную оценку.
                    def _lf(xx, la, mu, sg):
                        return np.log(lognormal_dndm(xx, np.exp(la), mu, sg))
                    p, _ = curve_fit(_lf, x, np.log(y), p0=(np.log(a0), mu0, sg0),
                                     bounds=([np.log(a0) - 15.0, lx.min() - 5.0, 1e-3],
                                             [np.log(a0) + 15.0, lx.max() + 5.0, 10.0]),
                                     maxfev=40000)
                    af, muf, sgf = float(np.exp(p[0])), float(p[1]), float(p[2])
                    meth = "least squares in log y"
                    rms = float(np.std(np.log(y) - _lf(x, *p)))
                else:
                    p, _ = curve_fit(lognormal_dndm, x, y, p0=(a0, mu0, sg0),
                                     bounds=([0.0, lx.min() - 5.0, 1e-3],
                                             [np.inf, lx.max() + 5.0, 10.0]),
                                     maxfev=20000)
                    af, muf, sgf = [float(v) for v in p]
                    meth = "least squares in linear y"
                    rms = float(np.std(np.log(y) - np.log(lognormal_dndm(x, af, muf, sgf))))
            except Exception as exc:
                af, muf, sgf, meth, rms = a0, mu0, sg0, "moment estimate", np.nan
                print("фрагментация: фит не сошёлся, беру моментную оценку:", exc)
            return af, muf, sgf, meth, rms, int(fm.sum()), x

        res = _fit(*GAUSS_GUARD)
        if res is None:
            print("гауссиана: у поколения %d меньше четырёх годных бинов -- пропускаю" % g_fit)
        else:
            a_f, mu_f, sg_f, method, rms, nb_fit, x_fit = res
            #  ОКОННАЯ СИСТЕМАТИКА.  Тот же фит, но с защитой на декаду шире.
            #  Пакет гауссиан только в СЕРДЦЕВИНЕ: хвосты -- это большие
            #  уклонения суммы g шагов, они не гауссовы, плюс у стока пакет
            #  обрезан.  Поэтому mu и sigma ползут с окном, плато нет, и
            #  называть sigma без этого числа нельзя.
            res2 = _fit(GAUSS_GUARD[0] + 1.0, GAUSS_GUARD[1])
            sg_win = abs(res2[2] - sg_f) if res2 is not None else np.nan
            mu_win = abs(res2[1] - mu_f) if res2 is not None else np.nan

            xl = np.logspace(np.log10(x_fit.min()), np.log10(x_fit.max()), 600)
            yl = lognormal_dndm(xl, a_f, mu_f, sg_f) * xl**2
            ax.loglog(xl, yl, "-", color="k", lw=1.4, zorder=5)
            y_all.append(yl)
            if GAUSS_LABEL:
                ax.text(GAUSS_LABEL_X, GAUSS_LABEL_Y, GAUSS_LABEL,
                        transform=ax.transAxes, ha="right", va="center",
                        fontsize=6.5, color="0.1", zorder=6, clip_on=True,
                        bbox=dict(facecolor="white", edgecolor="none", pad=0.8, alpha=0.85))
            gauss_info = (g_fit, nb_fit, method, a_f, mu_f, sg_f, rms,
                          mu_win, sg_win,
                          (np.log(m_inj) - mu_f) / g_fit, sg_f**2 / g_fit)

    _marks(ax, m_inj, m_sink)

    #  Подписан показатель СПЕКТРА.  Нарисовано m^2 dN/dm, её собственный
    #  наклон 2 + alpha, то есть кривая слабо РАСТЁТ с массой.
    pl = AN.spectrum_plateau(r, spec=s)
    ax.text(0.40, 0.72, r"$dN/dm \propto m^{-1.83}$",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=7, color="0.1")

    return np.concatenate([np.asarray(v)[np.isfinite(v) & (np.asarray(v) > 0)]
                           for v in y_all]), (m_inj, m_sink), pick, age, norm, gauss_info


# ======================================================================
#  ПАНЕЛЬ B -- коагуляция: спектр + изохроны + логнормаль
#  (блок PRL_STYLE_COMPENSATED_SPECTRUM из Analysis_open_coagulation_...)
# ======================================================================
def lognormal_dndm(x, area, mu, sigma):
    x = np.asarray(x, float)
    sigma = np.maximum(sigma, 1e-12)
    return area / (x * sigma * np.sqrt(2.0 * np.pi)) \
        * np.exp(-0.5 * ((np.log(x) - mu) / sigma) ** 2)


def panel_coagulation(fig, ax, cax):
    r, s = R_COAG, S_COAG
    pl_for_iso = GROW_COAG["pl"] if GROW_COAG is not None else None
    idx, tau, ncand = AN.pick_isochrones(r, N_CURVES, pl=pl_for_iso)
    groups = [np.atleast_1d(np.asarray(g, int)) for g in idx]
    if len(groups) == 0:
        raise RuntimeError("коагуляция: ни одной изохроны не выбрано")

    m = np.asarray(r["centers"], float)
    counts = np.asarray(r["iso_counts"], float)
    iso_dndm = np.asarray(r["iso_dndm"], float)
    nsnap_iso = max(float(r["iso_snapshots"]), 1.0)
    m_inj, m_sink = _scales(r)

    age = np.array([np.average(tau[g], weights=np.maximum(counts[g].sum(axis=1), 1e-300))
                    for g in groups])
    order = np.argsort(age)
    groups = [groups[i] for i in order]
    age = age[order]
    age_scaled = age / AGE_DIV_B

    y_all = []

    # ---- спектр -------------------------------------------------------
    ok_s = np.isfinite(s["F"]) & (s["F"] > 0) & np.isfinite(m) & (m > 0)
    ax.errorbar(m[ok_s], (s["F"] * m**2)[ok_s], yerr=(s["sigma"] * m**2)[ok_s],
                fmt="o", ms=2.0, mfc="white", mec="0.25", mew=0.5,
                ecolor="0.75", elinewidth=0.45, capsize=1.0, color="0.25", zorder=4)
    y_all.append((s["F"] * m**2)[ok_s])

    # ---- изохроны, цвет по возрасту -----------------------------------
    global LINES_B
    LINES_B, keep_age = [], []
    norm = LogNorm(vmin=max(age_scaled.min(), np.nextafter(0, 1)),
                   vmax=max(age_scaled.max(), age_scaled.min() * 1.0001))
    for g, tk in zip(groups, age_scaled):
        y = iso_dndm[g].sum(axis=0) / nsnap_iso
        cg = counts[g].sum(axis=0)
        ok = np.isfinite(y) & (y > 0) & (cg >= 3.0) & np.isfinite(m) & (m > 0)
        if not ok.any():
            continue
        h, = ax.loglog(m[ok], (y * m**2)[ok], "o", ms=2.0, markeredgewidth=0.0,
                       color=CMAP(norm(tk)), alpha=0.92, zorder=3)
        LINES_B.append(h); keep_age.append(tk)
        y_all.append((y * m**2)[ok])

    # ---- логнормальный фит по предпоследней изохроне -------------------
    fit_info = None
    age_scaled = np.asarray(keep_age, float)

    if SHOW_LOGNORMAL:
        FIT_GROUP = -2                      # -1 = самая последняя
        old = groups[FIT_GROUP]
        y_old = iso_dndm[old].sum(axis=0) / nsnap_iso
        cnt_old = counts[old].sum(axis=0)
        fm = np.isfinite(m) & np.isfinite(y_old) & (m > 0) & (y_old > 0) & (cnt_old >= 3.0)
        if fm.sum() < 3:
            raise RuntimeError("коагуляция: у выбранной изохроны меньше трёх годных бинов")
        x_fit, y_fit = m[fm], y_old[fm]
        try:
            widths = np.asarray(r["widths"], float)[fm]
        except Exception:
            widths = np.gradient(x_fit)
        w = np.maximum(y_fit * widths, 0.0)
        if not np.any(w > 0):
            w = y_fit / np.nanmax(y_fit)
        logx = np.log(x_fit)
        mu0 = float(np.average(logx, weights=w))
        sigma0 = max(float(np.sqrt(np.average((logx - mu0) ** 2, weights=w))), 0.15)
        area0 = float(np.sum(y_fit * widths))
        method = "moment estimate"
        try:
            from scipy.optimize import curve_fit
            popt, _ = curve_fit(lognormal_dndm, x_fit, y_fit, p0=(area0, mu0, sigma0),
                                bounds=([0.0, np.log(x_fit.min()) - 5.0, 1e-3],
                                        [np.inf, np.log(x_fit.max()) + 5.0, 10.0]),
                                maxfev=20000)
            area_fit, mu_fit, sigma_fit = [float(v) for v in popt]
            method = "nonlinear least squares"
        except Exception as exc:
            area_fit, mu_fit, sigma_fit = area0, mu0, sigma0
            print("коагуляция: SciPy недоступен, беру моментную оценку:", exc)

        x_line = np.logspace(np.log10(x_fit.min()), np.log10(x_fit.max()), 600)
        y_comp = lognormal_dndm(x_line, area_fit, mu_fit, sigma_fit) * x_line**2
        ax.loglog(x_line, y_comp, "-", color="k", lw=1.4, zorder=1.5)
        y_all.append(y_comp)
        #  Подпись ставится по x у пика, а по y -- в ДОЛЯХ РАМКИ.  В исходной
        #  тетради высота бралась от самого пика с жёстким потолком 1.2e7; на
        #  другом прогоне пик уезжает выше предела, и текст (он не обрезается
        #  по умолчанию) уходит рисоваться поверх СОСЕДНЕЙ панели.  Здесь этого
        #  случиться не может, а clip_on -- страховка.
        pk = int(np.nanargmax(y_comp))
        if LOGNORM_LABEL:
            ax.text(LOGNORM_LABEL_X, LOGNORM_LABEL_Y, LOGNORM_LABEL,
                    transform=ax.transAxes, ha="right", va="center",
                    fontsize=6.5, color="0.1", zorder=6, clip_on=True,
                    bbox=dict(facecolor="white", edgecolor="none", pad=0.8, alpha=0.85))
        fit_info = (age[FIT_GROUP], int(fm.sum()), method,
                    area_fit, mu_fit, sigma_fit)

    _marks(ax, m_inj, m_sink)

    pl = AN.spectrum_plateau(r, spec=s)
    ax.text(0.40, 0.62, r"$dN/dm \propto m^{-1.83}$",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=7, color="0.1")

    return np.concatenate([np.asarray(v)[np.isfinite(v) & (np.asarray(v) > 0)]
                           for v in y_all]), (m_inj, m_sink), age_scaled, fit_info, norm


def _auto_ylim(y):
    return (10.0 ** np.floor(np.log10(np.percentile(y, 1.0))),
            10.0 ** np.ceil(np.log10(y.max()) + 0.15))


# ======================================================================
#  РАСКЛАДКА  --  ОДНА стандартная одноколоночная фигура PRL
# ======================================================================
#  Обе панели живут внутри 3.375 x 2.85 дюйма, это ширина колонки PRL.
#  Что для этого пришлось сделать, и почему именно так:
#    * подпись оси y ОДНА на обе панели, по центру слева -- она у них
#      буквально одинаковая, дублировать её значит потратить поле дважды;
#    * подпись оси x одна, снизу: ось общая;
#    * цветовая шкала ОДНА на всю фигуру, если возраст меряется в t_c
#      (см. _age_scale).  Две шкалы в такой высоте не помещаются: их подписи
#      съедают больше места, чем сами панели.
#    * тиков по y по четыре на панель, иначе подписи сливаются.

AGE_DIV_A, LAB_A = _age_scale(R_FRAG)
AGE_DIV_B, LAB_B = _age_scale(R_COAG)
SHARED = (AGE_DIV_A is not None) and (AGE_DIV_B is not None) and (LAB_A == LAB_B)
if AGE_UNIT == "t_c" and not SHARED:
    print("t_c нет хотя бы у одного прогона -- откатываюсь на абсолютное время "
          "и две цветовые шкалы")
    AGE_DIV_A = AGE_DIV_B = 1e-6
    LAB_A = LAB_B = r"$\tau$  ($10^{-6}$)"

from matplotlib.ticker import LogLocator, NullLocator

with plt.rc_context(prl_rc):
    fig = plt.figure(figsize=FIGSIZE)
    #  hspace = 0: панели прижаты, ось x одна на двоих и подписана снизу.
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 0.040],
                          height_ratios=[1.0, 1.0], wspace=0.05, hspace=0.0)
    axA = fig.add_subplot(gs[0, 0])
    axB = fig.add_subplot(gs[1, 0], sharex=axA)
    if SHARED:
        cax = fig.add_subplot(gs[:, 1]);  caxA = caxB = cax
    else:
        caxA = fig.add_subplot(gs[0, 1]); caxB = fig.add_subplot(gs[1, 1])

    yA, (mi_A, ms_A), pick, ageA, normA, gaussA = panel_fragmentation(fig, axA, caxA)
    yB, (mi_B, ms_B), ageB, fit_info, normB = panel_coagulation(fig, axB, caxB)

    # ---- цветовая шкала ------------------------------------------------
    if SHARED:
        #  Одна шкала на обе панели: общий диапазон, общий цвет.  Обе панели
        #  перерисовываются под него -- иначе цвет на верхней панели значил бы
        #  не то же, что на нижней, а шкала стояла бы одна.
        lo = min(ageA.min(), ageB.min()); hi = max(ageA.max(), ageB.max())
        norm = LogNorm(vmin=lo, vmax=max(hi, lo * 1.0001))
        for coll_ages, ax_, lines in ((ageA, axA, LINES_A), (ageB, axB, LINES_B)):
            for h, tk in zip(lines, coll_ages):
                h.set_color(CMAP(norm(tk)))
        sm = plt.cm.ScalarMappable(cmap=CMAP, norm=norm); sm.set_array([])
        _age_ticks(fig.colorbar(sm, cax=cax), np.sort(np.concatenate([ageA, ageB])),
                   n_max=5, label=LAB_A)
    else:
        for cax_, ages_, lab_, nrm in ((caxA, ageA, LAB_A, normA),
                                       (caxB, ageB, LAB_B, normB)):
            sm = plt.cm.ScalarMappable(cmap=CMAP, norm=nrm); sm.set_array([])
            _age_ticks(fig.colorbar(sm, cax=cax_), ages_, n_max=3, label=lab_)

    # ---- общая ось масс -------------------------------------------------
    #  Коробки совпадают по диапазону, поток идёт навстречу: сверху справа
    #  налево, снизу слева направо.  Это и есть содержание рисунка.
    lo_m = min(mi_A, ms_A, mi_B, ms_B) / 2.5
    hi_m = max(mi_A, ms_A, mi_B, ms_B) * 2.5
    axA.set_xlim(*(XLIM if XLIM is not None else (lo_m, hi_m)))

    axA.set_ylim(*(YLIM_FRAG if YLIM_FRAG is not None else _auto_ylim(yA)))
    axB.set_ylim(*(YLIM_COAG if YLIM_COAG is not None else _auto_ylim(yB)))

    for ax in (axA, axB):
        ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=N_YTICKS))
        ax.yaxis.set_minor_locator(NullLocator())
    axA.tick_params(axis="x", which="both", labelbottom=False)
    axB.set_xlabel(r"$m$", labelpad=1)

    # ---- ОДНА подпись оси y на обе панели -------------------------------
    fig.text(0.012, 0.55, r"$m^{2}\,dN/dm$", rotation=90,
             ha="left", va="center", fontsize=8)

    # ---- подписи панелей ВНУТРИ рамки, не заголовками --------------------
    for ax, lab in zip((axA, axB), LABELS):
        ax.text(LABEL_X, LABEL_Y, lab, transform=ax.transAxes,
                ha="center", va="top", fontsize=7.5, color="0.1", zorder=7,
                bbox=dict(facecolor="white", edgecolor="none", pad=1.2, alpha=0.85))

    fig.subplots_adjust(left=0.155, right=0.885, bottom=0.135, top=0.985,
                        wspace=0.05, hspace=0.0)

    #  Подписи тиков цветовой шкалы центрируются на тике и торчат за край своей
    #  оси.  При двух шкалах и hspace = 0 нижняя подпись верхней садится на
    #  верхнюю подпись нижней -- поэтому шкалы поджимаются.  Панели не трогаем.
    for _cax in ({id(caxA): caxA, id(caxB): caxB}).values():
        _p = _cax.get_position()
        _pad = CBAR_PAD * _p.height
        _cax.set_position([_p.x0, _p.y0 + _pad, _p.width, _p.height - 2 * _pad])

    if SAVEFIG:
        fig.savefig(SAVEFIG, bbox_inches="tight")
        print("записано ->", SAVEFIG)

# ---- что нарисовано, числами ------------------------------------------
_u = "t_c" if SHARED and AGE_UNIT == "t_c" else "%g" % (AGE_DIV_A or 1.0)
print("фигура : %.3f x %.3f дюйма  (колонка PRL = 3.375)" % FIGSIZE)
print("верх   фрагментация : %s" % os.path.basename(FN_FRAG))
print("         g          : %s" % list(map(int, pick)))
print("         возраст    : %s   (единица %s)" % (["%.2f" % a for a in ageA], _u))
print("         инжекция %.3g -> сток %.3g   (поток справа налево)" % (mi_A, ms_A))
if gaussA is not None:
    print("         гауссиана  : поколение g = %d (GEN_FIT = %s), бинов %d, %s"
          % (gaussA[0], GEN_FIT, gaussA[1], gaussA[2]))
    print("                      mu = %+.4f +- %.4f (окно), sigma = %.4f +- %.4f (окно)"
          % (gaussA[4], gaussA[7], gaussA[5], gaussA[8]))
    print("                      медиана m = %.4g, остатки в ln y: rms = %.3f"
          % (np.exp(gaussA[4]), gaussA[6]))
    #  Два числа НА ПОКОЛЕНИЕ -- это моменты ОДНОГО шага дробления.  Считаны по
    #  gen_counts, то есть по ЧИСЛУ частиц; сверять их надо с
    #  AN.gen_moments(AN.generations(r, weight="number")), а не с "mass".
    print("                      (ln m_inj - mu)/g = %.4f, sigma^2/g = %.4f   "
          "-- моменты ОДНОГО шага" % (gaussA[9], gaussA[10]))
    if np.isfinite(gaussA[8]) and gaussA[8] > 0.1 * gaussA[5]:
        print("                      ВНИМАНИЕ: sigma едет на %.0f%% при сдвиге окна на"
              " декаду." % (100 * gaussA[8] / gaussA[5]))
        print("                      Пакет гауссиан только в сердцевине: хвосты -- это")
        print("                      большие уклонения суммы g шагов, они не гауссовы,")
        print("                      плюс у стока пакет обрезан.  Плато по окну нет,")
        print("                      поэтому sigma надо называть ВМЕСТЕ с окном.")
print("низ    коагуляция   : %s" % os.path.basename(FN_COAG))
print("         изохрон    : %d, возраст %s   (единица %s)"
      % (ageB.size, ["%.2f" % a for a in ageB], _u))
print("         инжекция %.3g -> сток %.3g   (поток слева направо)" % (mi_B, ms_B))
if fit_info is not None:
    print("         логнормаль : tau = %.4g, бинов %d, %s" % fit_info[:3])
    print("                      sigma = %.4g, медиана m = %.4g"
          % (fit_info[5], np.exp(fit_info[4])))
print("шкала возраста: %s" % ("ОДНА на обе панели" if SHARED else "две, по одной на панель"))
if not SAVEFIG:
    print("на диск ничего не писалось: SAVEFIG = None.")


## Логнормаль против нормали

Четыре кривые, две формы, один критерий. Верхний ряд — поколения
фрагментации, нижний — изохроны коагуляции.


In [ ]:
# ======================================================================
#  ЛОГНОРМАЛЬ ПРОТИВ НОРМАЛИ  --  четыре панели, один и тот же критерий
# ======================================================================
#  Вопрос простой: обязана ли форма быть логнормальной, или нормальное
#  распределение по САМОЙ m описало бы данные не хуже?
#
#  Проверяется в лоб.  К каждой кривой прикладываются ДВЕ функции:
#     lognormal   dN/dm = A /(m s sqrt(2pi)) exp[ -(ln m - mu)^2 / (2 s^2) ]
#     normal      dN/dm = A /(  s sqrt(2pi)) exp[ -(m    - M )^2 / (2 s^2) ]
#  Критерий ОДИН И ТОТ ЖЕ -- МНК в логарифме y, тот же набор бинов, тот же
#  порог.  Никакого преимущества ни одной из форм не даётся.
#
#  Почему ответ известен заранее и всё равно стоит его показать: дробление
#  МУЛЬТИПЛИКАТИВНО, каждый акт умножает массу на долю.  Логарифмы
#  складываются, центральная предельная теорема даёт гауссиану по ln m, то
#  есть логнормаль по m.  Нормаль по m получилась бы, если бы к массе на
#  каждом шаге ПРИБАВЛЯЛОСЬ независимое приращение.  Так не работает ни
#  дробление, ни слипание.
#
#  Верхний ряд -- поколения фрагментации, нижний -- изохроны коагуляции.

# ---- ЧТО СРАВНИВАТЬ --------------------------------------------------
CMP_GEN     = (14, 24)     # <<-- какие ПОКОЛЕНИЯ, верхний ряд
CMP_ISO     = (-2, -1)     # <<-- какие ИЗОХРОНЫ, нижний ряд: индексы в
                           # списке отобранных, отсортированном по возрасту.
                           # Взяты ДВЕ САМЫЕ СТАРЫЕ, и не для красоты:
                           # у молодых изохрон (tau/t_c ниже примерно 16) в
                           # спектре ещё сидит несхлопнувшийся пик инжекции на
                           # m = 1, и его не описывает НИ ОДНА из двух форм --
                           # rms там 1.5...2.4 у обеих.  Логнормальными
                           # изохроны становятся только после tau/t_c ~ 20.
                           # Поставьте (0, 1), чтобы это увидеть.
CMP_REBIN   = 3            # слить по стольку бинов, как на главной фигуре
CMP_MINCNT  = 30.0         # порог по СЫРЫМ отсчётам (поколения)
CMP_MINISO  = 3.0          # порог по отсчётам (изохроны, как в родной тетради)
CMP_FIGSIZE = (6.75, 4.4)  # ДВЕ колонки PRL: это диагностика, не главный рисунок
CMP_SAVEFIG = None         # напр. "fig_lognormal_vs_normal.pdf"; None = не писать

import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import curve_fit


def _logn(x, a, mu, s):
    """Гауссиана по ln m.  Она же логнормаль по m."""
    x = np.asarray(x, float); s = max(float(s), 1e-12)
    return a / (x * s * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((np.log(x) - mu) / s) ** 2)


def _norm(x, a, M, s):
    """Гауссиана по САМОЙ m.  Обычное нормальное распределение."""
    x = np.asarray(x, float); s = max(float(s), 1e-12)
    return a / (s * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((x - M) / s) ** 2)


def _rebin2(F, edges, k):
    n = (edges.size - 1) // k * k
    w = np.diff(edges)[:n]
    cnt = (np.asarray(F)[:n] * w).reshape(-1, k).sum(1)
    e2 = np.concatenate([edges[:n:k], [edges[n]]])
    return np.sqrt(e2[:-1] * e2[1:]), cnt / np.diff(e2), cnt


def _fit_both(x, y, w):
    """Оба фита по одним и тем же точкам, оба МНК в логарифме y.

    У нормали старт берётся из НЕСКОЛЬКИХ точек: её оптимум лежит далеко от
    любой разумной начальной догадки (фит уводит центр в отрицательные массы,
    чтобы левым хвостом изобразить степенной рост), и с одного старта
    оптимизатор туда не доходит -- получилось бы, что нормаль проиграла
    из-за плохого старта, а не по существу.
    """
    ly, lx = np.log(y), np.log(x)
    mu0 = float(np.average(lx, weights=w))
    s0 = max(float(np.sqrt(np.average((lx - mu0) ** 2, weights=w))), 0.15)
    a0 = float(w.sum())
    M0 = float(np.average(x, weights=w))
    sM0 = max(float(np.sqrt(np.average((x - M0) ** 2, weights=w))), 1e-6 * x.max())

    fL = lambda xx, la, mu, s: np.log(_logn(xx, np.exp(la), mu, s))
    pL, _ = curve_fit(fL, x, ly, p0=(np.log(a0), mu0, s0),
                      bounds=([np.log(a0) - 20, lx.min() - 5, 1e-3],
                              [np.log(a0) + 20, lx.max() + 5, 10.0]), maxfev=80000)
    rL = float(np.std(ly - fL(x, *pL))); eL = float(np.abs(ly - fL(x, *pL)).max())

    fN = lambda xx, la, M, s: np.log(_norm(xx, np.exp(la), M, s))
    best = None
    for M_try in (x.min(), np.exp(mu0), M0, x.max()):
        for s_try in (sM0, np.exp(mu0), x.max() / 2.0, x.max() * 2.0):
            try:
                p, _ = curve_fit(fN, x, ly, p0=(np.log(a0), M_try, s_try),
                                 bounds=([np.log(a0) - 30, -5 * x.max(), 1e-9],
                                         [np.log(a0) + 30, 5 * x.max(), 100 * x.max()]),
                                 maxfev=100000)
                rr = float(np.std(ly - fN(x, *p)))
                if np.isfinite(rr) and (best is None or rr < best[1]):
                    best = (p, rr, float(np.abs(ly - fN(x, *p)).max()))
            except Exception:
                pass
    if best is None:
        return (pL, rL, eL), None
    return (pL, rL, eL), best


# ---- собрать четыре набора точек --------------------------------------
CURVES = []

ed = np.asarray(R_FRAG["edges"], float)
H = np.asarray(R_FRAG["gen_counts"], float)
nsn = max(float(R_FRAG["gen_snapshots"]), 1.0)
for gg in CMP_GEN:
    mr, Fr_raw, cnt = _rebin2(H[int(gg)] / np.diff(ed), ed, CMP_REBIN)
    Fr = Fr_raw / nsn
    ok = np.isfinite(mr) & np.isfinite(Fr) & (mr > 0) & (Fr > 0) & (cnt >= CMP_MINCNT)
    if ok.sum() < 4:
        print("сравнение: у поколения %d меньше четырёх годных бинов -- пропускаю" % gg)
        continue
    CURVES.append(("Fragmentation", r"$g=%d$" % int(gg), mr[ok], Fr[ok], cnt[ok]))

_pl = GROW_COAG["pl"] if GROW_COAG is not None else None
_idx, _tau, _ = AN.pick_isochrones(R_COAG, N_CURVES, pl=_pl)
_grp = [np.atleast_1d(np.asarray(q, int)) for q in _idx]
_cnts = np.asarray(R_COAG["iso_counts"], float)
_iso = np.asarray(R_COAG["iso_dndm"], float)
_nsi = max(float(R_COAG["iso_snapshots"]), 1.0)
_mc = np.asarray(R_COAG["centers"], float)
_ag = np.array([np.average(_tau[q], weights=np.maximum(_cnts[q].sum(axis=1), 1e-300))
                for q in _grp])
_o = np.argsort(_ag); _grp = [_grp[i] for i in _o]; _ag = _ag[_o]
_tc = (R_COAG["meta"].get("analysis", {}) or {}).get("t_c")
for kk in CMP_ISO:
    q = _grp[int(kk)]
    y = _iso[q].sum(axis=0) / _nsi
    c = _cnts[q].sum(axis=0)
    ok = np.isfinite(_mc) & np.isfinite(y) & (_mc > 0) & (y > 0) & (c >= CMP_MINISO)
    if ok.sum() < 4:
        print("сравнение: у изохроны %s меньше четырёх годных бинов -- пропускаю" % kk)
        continue
    lab = (r"$\tau/t_c=%.1f$" % (_ag[int(kk)] / float(_tc))) if _tc else \
          (r"$\tau=%.3g$" % _ag[int(kk)])
    CURVES.append(("Coagulation", lab, _mc[ok], y[ok], c[ok]))

# ---- фит и рисунок ----------------------------------------------------
prl_rc2 = {"figure.dpi": 160, "font.size": 8, "axes.labelsize": 8,
           "legend.fontsize": 6.0, "xtick.labelsize": 7, "ytick.labelsize": 7,
           "axes.linewidth": 0.8, "xtick.direction": "in", "ytick.direction": "in",
           "xtick.top": True, "ytick.right": True, "axes.grid": False}

rows = ["Fragmentation", "Coagulation"]
REPORT = []
with plt.rc_context(prl_rc2):
    figc, axc = plt.subplots(2, 2, figsize=CMP_FIGSIZE, sharey=False)
    for ax_, (proc, lab, x, y, w) in zip(axc.ravel(), CURVES):
        (pL, rL, eL), N = _fit_both(x, y, w)
        xl = np.logspace(np.log10(x.min()), np.log10(x.max()), 800)
        ax_.loglog(x, y * x**2, "o", ms=2.6, mfc="white", mec="0.25", mew=0.6,
                   zorder=4, label="data")
        ax_.loglog(xl, _logn(xl, np.exp(pL[0]), pL[1], pL[2]) * xl**2, "-",
                   color="k", lw=1.4, zorder=3,
                   label="lognormal, rms %.2f" % rL)
        if N is not None:
            pN, rN, eN = N
            ax_.loglog(xl, _norm(xl, np.exp(pN[0]), pN[1], pN[2]) * xl**2, "--",
                       color="crimson", lw=1.4, zorder=2,
                       label="normal, rms %.2f" % rN)
        else:
            pN = (np.nan,) * 3; rN = eN = np.nan
        ax_.set_xlim(x.min() / 2.0, x.max() * 2.0)
        yy = np.concatenate([y * x**2,
                             _logn(xl, np.exp(pL[0]), pL[1], pL[2]) * xl**2])
        yy = yy[np.isfinite(yy) & (yy > 0)]
        ax_.set_ylim(10.0 ** np.floor(np.log10(yy.min())),
                     10.0 ** np.ceil(np.log10(yy.max()) + 0.2))
        ax_.text(0.03, 0.96, "%s\n%s" % (proc, lab), transform=ax_.transAxes,
                 ha="left", va="top", fontsize=7, color="0.1",
                 bbox=dict(facecolor="white", edgecolor="none", pad=1.2, alpha=0.85))
        ax_.legend(frameon=False, loc="lower center", handlelength=1.5,
                   borderpad=0.15, labelspacing=0.25)
        ax_.set_xlabel(r"$m$"); ax_.set_ylabel(r"$m^{2}\,dN/dm$")
        REPORT.append((proc, lab, int(x.size), pL, rL, eL, pN, rN, eN))
    figc.tight_layout(pad=0.5)
    if CMP_SAVEFIG:
        figc.savefig(CMP_SAVEFIG, bbox_inches="tight")
        print("записано ->", CMP_SAVEFIG)

try:
    from IPython.display import display
    display(figc); plt.close(figc)
except Exception:
    plt.show()

# ---- числа -------------------------------------------------------------
print("ЛОГНОРМАЛЬ ПРОТИВ НОРМАЛИ.  Один критерий: МНК в логарифме y, одни бины.")
print("%-14s %-16s %5s | %9s %8s %7s | %11s %10s %7s | %6s"
      % ("процесс", "кривая", "бинов", "mu", "sigma", "rms", "M", "s", "rms", "хуже"))
for proc, lab, nb_, pL, rL, eL, pN, rN, eN in REPORT:
    ratio = (rN / rL) if np.isfinite(rN) and rL > 0 else np.nan
    print("%-14s %-16s %5d | %+9.4f %8.4f %7.3f | %11.4g %10.4g %7.3f | %5.1fx"
          % (proc, lab.replace("$", "").replace("\\tau", "tau").replace("_c", "c"),
             nb_, pL[1], pL[2], rL, pN[1], pN[2], rN, ratio))
    if rL > 1.0:
        print("%36s ^^ ОБЕ формы плохи (rms логнормали %.2f).  Это не довод в"
              % ("", rL))
        print("%36s    пользу нормали: кривая просто не описывается ни одной" % "")
        print("%36s    гладкой формой -- в ней сидит пик инжекции." % "")
    elif np.isfinite(pN[1]) and pN[1] < 0:
        print("%36s ^^ центр нормали УШЁЛ В ОТРИЦАТЕЛЬНЫЕ МАССЫ: у неё нет формы,"
              % "")
        print("%36s    способной расти на декадах, и единственный выход --"
              % "")
        print("%36s    изобразить рост куском левого хвоста." % "")
_sg = [rp[3][2] for rp in REPORT if rp[0] == "Coagulation" and rp[4] < 1.0]
if len(_sg) > 1:
    print()
    print("ШИРИНА чистых изохрон коагуляции: sigma = %s"
          % ", ".join("%.3f" % v for v in _sg))
    print("Она НЕ меняется с возрастом, хотя типичная масса за это время выросла")
    print("в разы.  Это и есть самоподобие: пакет едет по ln m, не расплываясь.")
print()
print("mu и sigma логнормали -- по ln m.  M и s нормали -- по самой m,")
print("поэтому колонки НЕ сравниваются между собой; сравниваются только rms.")


## Коллапс на $m/\langle m\rangle$

Без фита. Среднее и стандартное отклонение считаются прямо из гистограммы,
по самой $m$, и на них делится ось.


In [ ]:
# ======================================================================
#  КОЛЛАПС НА  m / <m>  --  БЕЗ ФИТА
# ======================================================================
#  Никаких подгонок.  Для каждой кривой берутся ДВА ЧИСЛА, посчитанные
#  прямо из гистограммы, весами служат сырые отсчёты:
#
#      <m>   = sum(m * cnt) / sum(cnt)
#      std   = sqrt( sum((m - <m>)^2 * cnt) / sum(cnt) )
#
#  Моменты берутся по САМОЙ m, не по ln m.  Дальше ось заменяется, и у
#  панелей ПО-РАЗНОМУ:
#
#      ФРАГМЕНТАЦИЯ, верх:   u = (m - <m>) / std      ось ЛИНЕЙНАЯ
#      КОАГУЛЯЦИЯ,   низ:    u =  m / <m>             ось ЛОГАРИФМИЧЕСКАЯ
#
#  Сверху ось обязана быть линейной: u уходит в минус, логарифма от неё нет.
#  И всё.  Ни одного параметра, ни одной итерации оптимизатора.
#
#  ЧТО ЭТО ПРОВЕРЯЕТ.  Если распределение самоподобно, то есть имеет вид
#  n(m) = <m>^(-2) * Phi(m/<m>), то отношение std/<m> -- ЧИСТОЕ ЧИСЛО, одно
#  и то же для всех кривых, потому что оно и есть второй момент функции Phi.
#  Значит проверка самоподобия делается вообще без картинки: посмотреть на
#  столбец std/<m>.  Постоянен -- коллапс обязан получиться.  Едет -- никакое
#  масштабирование одной величиной кривые не сложит, и это не вопрос выбора
#  переменной, а свойство задачи.
#
#  Кривые рисуются серыми линиями, от светлой к тёмной по возрасту.

# ---- ЧТО РИСОВАТЬ ----------------------------------------------------
Z_GEN     = None        # None = те же поколения, что на главной фигуре (GEN_PICK)
Z_ISO     = None        # None = все отобранные изохроны
Z_Y       = "m1"        # "m1" = m dN/dm  |  "m2" = m^2 dN/dm
Z_SCALE   = "none"      # "none" -- НИЧЕГО не нормировать, высоты свои
                        # "peak" -- поделить на СОБСТВЕННЫЙ максимум данных
                        #           (не фита -- фита здесь нет)
Z_XVAR_FRAG  = "std"    # ВЕРХНЯЯ панель, переменная:
                        #   "std"   -> (m - <m>)/sigma
                        #   "ratio" -> m/<m>,  как внизу; тогда ось чисто
                        #              логарифмическая, потому что отношение
                        #              всегда положительно
Z_XSCALE_FRAG = "symlog"  # ВЕРХНЯЯ панель, шкала: "linear" | "symlog" | "log"
                        #   "log" работает ТОЛЬКО с Z_XVAR_FRAG = "ratio":
                        #   у (m - <m>)/sigma есть отрицательная часть, и
                        #   логарифма от неё не существует.  "symlog" -- выход:
                        #   линейна в окрестности нуля, логарифмическая дальше,
                        #   и в обе стороны.
Z_LINTHRESH = 0.3       # полуширина линейного участка symlog
Z_XLIM_FRAG = None      # None = автоматически по данным
Z_XLIM_COAG = (1e-3, 5e1)     # пределы по m/<m>, НИЗ (ось логарифмическая)
Z_COLOR   = "gray"      # "gray" -- серая гамма по возрасту | "age" -- viridis
Z_FIGSIZE = (3.375, 3.30)
Z_SAVEFIG = None

import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

_gen = tuple(GEN_PICK) if Z_GEN is None else tuple(Z_GEN)
_pw = 1 if Z_Y == "m1" else 2


def _moments(x, w):
    """<m> и std ПО САМОЙ m.  Веса -- сырые отсчёты, то есть число частиц."""
    mean = float(np.average(x, weights=w))
    var = float(np.average((x - mean) ** 2, weights=w))
    return mean, float(np.sqrt(max(var, 0.0)))


# ---- собрать кривые ---------------------------------------------------
ROWS = [[], []]                    # 0 -- фрагментация, 1 -- коагуляция

_ed = np.asarray(R_FRAG["edges"], float)
_H = np.asarray(R_FRAG["gen_counts"], float)
_ns = max(float(R_FRAG["gen_snapshots"]), 1.0)
_tg = np.asarray(R_FRAG["gen_tau"], float)
_dvA, _ = _age_scale(R_FRAG)
for gg in _gen:
    gg = int(gg)
    mr, Fr_raw, cnt = _rebin(_H[gg] / np.diff(_ed), _ed, GEN_REBIN)
    Fr = Fr_raw / _ns
    ok = np.isfinite(mr) & np.isfinite(Fr) & (mr > 0) & (Fr > 0) & (cnt >= MIN_CNT)
    if ok.sum() >= 5:
        ROWS[0].append((r"$g=%d$" % gg, mr[ok], Fr[ok], cnt[ok], _tg[gg] / (_dvA or 1.0)))

_pl = GROW_COAG["pl"] if GROW_COAG is not None else None
_ix, _tau, _ = AN.pick_isochrones(R_COAG, N_CURVES, pl=_pl)
_gp = [np.atleast_1d(np.asarray(q, int)) for q in _ix]
_cc = np.asarray(R_COAG["iso_counts"], float)
_dd = np.asarray(R_COAG["iso_dndm"], float)
_nsi = max(float(R_COAG["iso_snapshots"]), 1.0)
_mc = np.asarray(R_COAG["centers"], float)
_ages = np.array([np.average(_tau[q], weights=np.maximum(_cc[q].sum(axis=1), 1e-300))
                  for q in _gp])
_o = np.argsort(_ages); _gp = [_gp[i] for i in _o]; _ages = _ages[_o]
_dvB, _ = _age_scale(R_COAG)
for kk in (range(len(_gp)) if Z_ISO is None else [int(v) for v in Z_ISO]):
    q = _gp[kk]
    y = _dd[q].sum(axis=0) / _nsi
    c = _cc[q].sum(axis=0)
    ok = np.isfinite(_mc) & np.isfinite(y) & (_mc > 0) & (y > 0) & (c >= 3.0)
    if ok.sum() >= 5:
        ROWS[1].append((r"$\tau=%.1f$" % (_ages[kk] / (_dvB or 1.0)),
                        _mc[ok], y[ok], c[ok], _ages[kk] / (_dvB or 1.0)))

# ---- рисунок ----------------------------------------------------------
prl_rc3 = {"figure.dpi": 160, "font.size": 8, "axes.labelsize": 8,
           "legend.fontsize": 6.0, "xtick.labelsize": 7, "ytick.labelsize": 7,
           "axes.linewidth": 0.8, "xtick.direction": "in", "ytick.direction": "in",
           "xtick.top": True, "ytick.right": True, "axes.grid": False}
ZREP = []
_uu = [[], []]
with plt.rc_context(prl_rc3):
    figz, axz = plt.subplots(2, 1, figsize=Z_FIGSIZE)
    _allage = np.array([c[4] for r in ROWS for c in r])
    znorm = LogNorm(vmin=_allage.min(), vmax=max(_allage.max(), _allage.min() * 1.0001))
    for row, (ax_, name) in enumerate(zip(axz, ("Fragmentation", "Coagulation"))):
        for lab, x, y, w, age in ROWS[row]:
            mean, std = _moments(x, w)
            #  Сверху центрируем и делим на std, снизу только делим на <m>:
            #  снизу ширина постоянна, вычитать нечего.
            if row == 0:
                u = (x / mean) if Z_XVAR_FRAG == "ratio" else ((x - mean) / std)
            else:
                u = x / mean
            _uu[row].append(u)
            val = y * x ** _pw
            if Z_SCALE == "peak":
                val = val / np.nanmax(val)
            #  Серая гамма: светлое -- молодое, тёмное -- старое.  Ровно тот же
            #  порядок, что и цвет на главной фигуре, только без цвета.
            if Z_COLOR == "gray":
                col = str(0.78 - 0.70 * float(np.clip(znorm(age), 0.0, 1.0)))
            else:
                col = CMAP(znorm(age))
            ax_.plot(u, val, "-", lw=1.0, color=col, zorder=3)
            ZREP.append((name, lab, int(x.size), mean, std, std / mean,
                         float(x.max() / mean)))
        ax_.set_yscale("log")
        if row == 0:
            _ratio = (Z_XVAR_FRAG == "ratio")
            _sc = Z_XSCALE_FRAG
            if _sc == "log" and not _ratio:
                #  Молча подменять шкалу нельзя: тогда с картинки пропала бы
                #  вся часть кривой ниже среднего, а это БОЛЬШИНСТВО бинов.
                print("ВЕРХ: Z_XSCALE_FRAG = 'log' несовместимо с "
                      "Z_XVAR_FRAG = 'std' -- (m-<m>)/sigma отрицательна слева "
                      "от среднего.  Беру 'symlog'.")
                _sc = "symlog"
            if _sc == "log":
                ax_.set_xscale("log")
            elif _sc == "symlog":
                ax_.set_xscale("symlog", linthresh=Z_LINTHRESH,
                               linscale=0.6, subs=[2, 3, 5])
            _lo = min(v.min() for v in _uu[row]); _hi = max(v.max() for v in _uu[row])
            if Z_XLIM_FRAG is not None:
                ax_.set_xlim(*Z_XLIM_FRAG)
            elif _sc == "linear":
                _pad = 0.06 * (_hi - _lo); ax_.set_xlim(_lo - _pad, _hi + _pad)
            else:
                ax_.set_xlim(_lo * 1.4 if _lo < 0 else _lo / 1.4, _hi * 1.4)
            ax_.axvline(1.0 if _ratio else 0.0, color="0.1", lw=0.6, ls=":", zorder=1)
            ax_.set_xlabel(r"$m/\langle m\rangle$" if _ratio
                           else r"$(m-\langle m\rangle)/\sigma$", labelpad=1)
        else:
            ax_.set_xscale("log"); ax_.set_xlim(*Z_XLIM_COAG)
            ax_.axvline(1.0, color="0.1", lw=0.6, ls=":", zorder=1)
            ax_.set_xlabel(r"$m/\langle m\rangle$", labelpad=1)
        ax_.text(0.03, 0.94, name, transform=ax_.transAxes, ha="left", va="top",
                 fontsize=7.5, color="0.1", zorder=6,
                 bbox=dict(facecolor="white", edgecolor="none", pad=1.2, alpha=0.85))
    _yl = (r"$m\,dN/dm$" if _pw == 1 else r"$m^{2}\,dN/dm$") + \
          ("" if Z_SCALE == "none" else "   (peak $=1$)")
    figz.text(0.012, 0.55, _yl, rotation=90, ha="left", va="center", fontsize=8)
    figz.subplots_adjust(left=0.180, right=0.975, bottom=0.105, top=0.985, hspace=0.42)
    if Z_SAVEFIG:
        figz.savefig(Z_SAVEFIG, bbox_inches="tight"); print("записано ->", Z_SAVEFIG)

try:
    from IPython.display import display
    display(figz); plt.close(figz)
except Exception:
    plt.show()

# ---- числа -------------------------------------------------------------
print("МОМЕНТЫ ПО САМОЙ m, БЕЗ ФИТА.  Веса -- сырые отсчёты.")
print("%-14s %-10s %5s %12s %12s %9s %10s"
      % ("процесс", "кривая", "бинов", "<m>", "std", "std/<m>", "m_max/<m>"))
for nm, lab, nb_, mean, std, cv, top in ZREP:
    print("%-14s %-10s %5d %12.4g %12.4g %9.3f %10.1f"
          % (nm, lab.replace("$", "").replace("\\tau", "tau"), nb_, mean, std, cv, top))

print()
for nm in ("Coagulation", "Fragmentation"):
    cv = np.array([r[5] for r in ZREP if r[0] == nm])
    if cv.size < 2:
        continue
    spread = (cv.max() - cv.min()) / cv.mean()
    print("%-14s std/<m> = %s" % (nm, ", ".join("%.2f" % v for v in cv)))
    print("%-14s разброс %.0f%% вокруг %.2f -- %s"
          % ("", 100 * spread, cv.mean(),
             "ПОСТОЯННО, коллапс на m/<m> обязан работать" if spread < 0.35
             else "ЕДЕТ, одной величиной эти кривые не сложить"))
print()
print("std/<m> -- это второй момент масштабной функции Phi, чистое число.")
print("Оно постоянно тогда и только тогда, когда n(m) = <m>^(-2) Phi(m/<m>).")
print("Никакая замена переменной этого не чинит: если оно едет, формы РАЗНЫЕ.")


## Форма сохраняется: коллапс с точными кумулянтами

Та же процедура, что в последних ячейках `Split_generations.ipynb`, но на
данных прогона. $\mu$ и $\sigma$ — интеграл от закона дробления, не фит.


In [ ]:
# ======================================================================
#  ФОРМА СОХРАНЯЕТСЯ.  Коллапс, как в Split_generations
# ======================================================================
#  Верх -- ПОКОЛЕНИЯ фрагментации.  Ровно та же процедура, что в последних
#  ячейках Split_generations, только теперь на данных ПРОГОНА, а не на
#  игрушечном дереве:
#
#      z = ( ln(m/m_inj) - mu*g ) / ( sigma*sqrt(g) )
#
#  mu и sigma -- ТОЧНЫЕ кумулянты одного дробления, взятые интегралом от
#  закона дробления движка.  НИЧЕГО НЕ ПОДГОНЯЕТСЯ.  Это и делает картинку
#  утверждением: если бы mu и sigma подбирались по каждому поколению,
#  кривые сошлись бы по построению и не значили бы ничего.
#
#  Закон дробления движка (строка 1682 BF_warm_start_v5):
#      xi = 0.5 + w*(2*rnd() - 1),   куски  xi*m  и  (1-xi)*m
#  то есть при w = 0.5 это равномерная xi на (0,1).  frag_min_ratio решает,
#  СЛУЧИТСЯ ли акт, но xi не режет -- проверено по коду.  gen_counts считает
#  ВСЕ живые частицы поколения g, то есть полное дерево: правило "piece",
#  mu = -1, sigma^2 = 1.
#
#  НОРМИРОВКА БЕЗ ЕДИНОГО ПАРАМЕТРА.  Коробка обрезает пакет: снизу стоком,
#  сверху инжекцией.  Поэтому у каждого поколения видна СВОЯ доля кривой --
#  от 98% при g = 6 до 2% при g = 24.  Но mu и sigma известны точно, значит
#  видимая доля СЧИТАЕТСЯ:
#
#      frac(g) = Phi(z_инж) - Phi(z_сток)
#
#  и плотность восстанавливается как  (cnt/sum cnt) * frac / dz.  Никакой
#  подгонки по вертикали: каждая кривая встаёт на своё место сама.
#
#  Именно поэтому поколения РАЗНЫЕ куски одной кривой и покрывают вместе
#  больше, чем любое поодиночке: малые g дают левое крыло и центр, большие --
#  правый хвост, куда малым не дотянуться по статистике.
#
#  Низ -- ИЗОХРОНЫ коагуляции на m/<m>.  Там ширина не растёт, снимать надо
#  только положение, и <m> считается прямо из гистограммы.

# ---- ЧТО РИСОВАТЬ ----------------------------------------------------
CL_GEN    = (1, 2, 3, 4, 6, 8, 11, 14, 18, 24)   # <<-- поколения, верх
CL_ISO    = None        # None = все отобранные изохроны, низ
CL_REBIN  = 2           # слить по стольку бинов (0.1 dex -> 0.2 dex)
CL_MINCNT = 30.0        # порог по СЫРЫМ отсчётам на бин
CL_RULE   = "piece"     # правило шага: gen_counts считает ВСЕ частицы -> "piece".
                        # "matter" -- если считать трейсер по массе.
CL_TILT   = True        # ПОПРАВКА НА ВРЕМЯ ПРЕБЫВАНИЯ.  См. пояснение ниже.
CL_B      = None        # b для этой поправки.  None = взять из ЗАКОНА ОЖИДАНИЯ
                        # этого же прогона, то есть из независимого измерения.
CL_EDGE   = True        # пунктиром: Эджворт с ТОЧНОЙ асимметрией, для среднего g
CL_LOGY   = True        # логарифмический y: без него не видно хвостов
CL_XLIM   = (-4.0, 4.5)
CL_FIGSIZE = (3.375, 3.30)
CL_SAVEFIG = None

import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from math import erf

_PHI = np.vectorize(lambda t: 0.5 * (1.0 + erf(t / np.sqrt(2.0))))
_phi = lambda z: np.exp(-0.5 * np.asarray(z) ** 2) / np.sqrt(2 * np.pi)


def _step_cumulants(w, rule, n=400001):
    """ТОЧНЫЕ mu, sigma^2, gamma1 одного дробления -- интегралом, не подгонкой.
    Та же функция, что step_cumulants в Split_generations."""
    xi = np.linspace(0.5 - w, 0.5 + w, n)[1:-1]
    tz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    v, p = ((xi, np.ones_like(xi)) if rule == "piece" else
            (xi, xi) if rule == "matter" else
            (np.maximum(xi, 1 - xi), np.ones_like(xi)) if rule == "heavy" else
            (np.minimum(xi, 1 - xi), np.ones_like(xi)))
    p = p / tz(p, xi); l = np.log(v)
    mu = float(tz(p * l, xi))
    m2 = float(tz(p * (l - mu) ** 2, xi))
    m3 = float(tz(p * (l - mu) ** 3, xi))
    return mu, m2, m3 / m2 ** 1.5


#  ПОЧЕМУ НУЖЕН СДВИГ НА sqrt(g)/b.
#  Снимок -- это НЕ ветвящийся процесс.  Это ветвящийся процесс, взвешенный на
#  время стоянки: частица сидит на своей массе время ~ m^(1/b) = exp(x/b), и
#  тяжёлые попадают в снимок чаще лёгких.  Экспоненциальный наклон гауссианы
#  на exp(x/b) сдвигает её среднее на sigma^2 g / b, что в единицах z равно
#
#      sqrt(g) / b
#
#  Это НЕ подгонка: b берётся из закона ожидания ЭТОГО ЖЕ прогона, то есть из
#  измерения, которое к поколениям отношения не имеет.  Проверка честная:
#  если поправка верна, коллапс должен стать ЛУЧШЕ, а если это просто лишний
#  параметр -- разницы не будет.  На этих данных медиана Колмогорова падает
#  с 0.159 до 0.091, то есть почти вдвое.
#
#  И знак важен.  Асимметрия дробления gamma1 = -2 сдвигала бы кривую ВЛЕВО,
#  а данные уехали ВПРАВО.  Значит наблюдаемое отклонение от гауссианы -- не
#  третий кумулянт, а именно наклон по времени стоянки.  Эджворт нарисован
#  пунктиром, чтобы это было видно: он идёт в другую сторону.

_W = float(R_FRAG["meta"].get("frag_split_width", 0.5))
MU_S, VAR_S, GAM_S = _step_cumulants(_W, CL_RULE)
_mi = float(R_FRAG["meta"]["injection_mass"])
_ms = float(R_FRAG["meta"]["sink_mass"])

_B = CL_B
if CL_TILT and _B is None:
    try:
        _B = 1.0 / AN.waiting_law(R_FRAG)["p"]
    except Exception as _e:
        _B = None
        print("закон ожидания недоступен, поправку на время пребывания выключаю:", _e)
_TILT = (CL_TILT and _B is not None and np.isfinite(_B) and _B > 0)

# ---- ВЕРХ: поколения --------------------------------------------------
_ed = np.asarray(R_FRAG["edges"], float)
_H = np.asarray(R_FRAG["gen_counts"], float)
_tg = np.asarray(R_FRAG["gen_tau"], float)
_dvA, _ = _age_scale(R_FRAG)


def _reb_counts(c, edges, k):
    """Слить по k бинов, вернув (центры, отсчёты, ширины в ln m)."""
    n = (edges.size - 1) // k * k
    e2 = np.concatenate([edges[:n:k], [edges[n]]])
    return (np.sqrt(e2[:-1] * e2[1:]), np.asarray(c)[:n].reshape(-1, k).sum(1),
            np.diff(np.log(e2)))


GENC, CREP = [], []
for g in CL_GEN:
    g = int(g)
    if g >= _H.shape[0] or _H[g].sum() <= 0:
        continue
    mc, cnt, dlnm = _reb_counts(_H[g], _ed, CL_REBIN)
    tot = float(cnt.sum())
    if tot <= 0:
        continue
    s_g = np.sqrt(VAR_S * g)
    _sh = (np.sqrt(g) / _B) if _TILT else 0.0        # наклон по времени стоянки
    z = (np.log(mc / _mi) - MU_S * g) / s_g - _sh
    dz = dlnm / s_g
    z_lo = (np.log(_ms / _mi) - MU_S * g) / s_g - _sh   # сток
    z_hi = (0.0 - MU_S * g) / s_g - _sh                 # инжекция
    frac = float(_PHI(z_hi) - _PHI(z_lo))
    p = cnt / tot * frac / dz                        # <-- ни одного параметра
    ok = np.isfinite(p) & (p > 0) & (cnt >= CL_MINCNT)
    if ok.sum() < 4:
        continue
    GENC.append((g, z[ok], p[ok], _tg[g] / (_dvA or 1.0)))
    #  Колмогоров считается по ВИДИМОМУ окну, поэтому сравнивать надо с
    #  Phi, перенормированной на то же окно -- иначе обрезание засчиталось бы
    #  как отклонение формы.
    #  ВЗВЕШЕННАЯ эмпирическая функция распределения.  Разворачивать бины в
    #  отдельные отсчёты нельзя: в бине бывают миллионы частица-снимков, и
    #  np.repeat съедает всю память.  Накопленная сумма весов даёт то же самое.
    _o2 = np.argsort(z[ok]); _zs = z[ok][_o2]; _ws = cnt[ok][_o2]
    _F = np.cumsum(_ws) / _ws.sum()
    _Fth = (_PHI(_zs) - _PHI(z_lo)) / max(frac, 1e-300)
    CREP.append((g, int(ok.sum()), frac, float(np.max(np.abs(_F - _Fth)))))

# ---- НИЗ: изохроны ----------------------------------------------------
_pl = GROW_COAG["pl"] if GROW_COAG is not None else None
_ix, _tau, _ = AN.pick_isochrones(R_COAG, N_CURVES, pl=_pl)
_gp = [np.atleast_1d(np.asarray(q, int)) for q in _ix]
_cc = np.asarray(R_COAG["iso_counts"], float)
_dd = np.asarray(R_COAG["iso_dndm"], float)
_nsi = max(float(R_COAG["iso_snapshots"]), 1.0)
_mc2 = np.asarray(R_COAG["centers"], float)
_ages = np.array([np.average(_tau[q], weights=np.maximum(_cc[q].sum(axis=1), 1e-300))
                  for q in _gp])
_o = np.argsort(_ages); _gp = [_gp[i] for i in _o]; _ages = _ages[_o]
_dvB, _ = _age_scale(R_COAG)
ISOC, IREP = [], []
for kk in (range(len(_gp)) if CL_ISO is None else [int(v) for v in CL_ISO]):
    q = _gp[kk]
    y = _dd[q].sum(axis=0) / _nsi
    c = _cc[q].sum(axis=0)
    ok = np.isfinite(_mc2) & np.isfinite(y) & (_mc2 > 0) & (y > 0) & (c >= 3.0)
    if ok.sum() < 5:
        continue
    x, yy, w_ = _mc2[ok], y[ok], c[ok]
    mean = float(np.average(x, weights=w_))
    std = float(np.sqrt(np.average((x - mean) ** 2, weights=w_)))
    val = yy * x                                     # m dN/dm = плотность по ln m
    ISOC.append((_ages[kk] / (_dvB or 1.0), x / mean, val / np.nanmax(val)))
    IREP.append((_ages[kk] / (_dvB or 1.0), int(ok.sum()), mean, std / mean))

# ---- рисунок ----------------------------------------------------------
prl_rc4 = {"figure.dpi": 160, "font.size": 8, "axes.labelsize": 8,
           "legend.fontsize": 5.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
           "axes.linewidth": 0.8, "xtick.direction": "in", "ytick.direction": "in",
           "xtick.top": True, "ytick.right": True, "axes.grid": False}
with plt.rc_context(prl_rc4):
    figc2, axc2 = plt.subplots(2, 1, figsize=CL_FIGSIZE)

    # --- верх ---
    ax0 = axc2[0]
    _ag = np.array([c[3] for c in GENC])
    nA = LogNorm(vmin=_ag.min(), vmax=max(_ag.max(), _ag.min() * 1.0001))
    for g, z, p, age in GENC:
        ax0.plot(z, p, "-", lw=0.9, color=CMAP(nA(age)), alpha=0.95, zorder=3)
    zz = np.linspace(CL_XLIM[0], CL_XLIM[1], 500)
    ax0.plot(zz, _phi(zz), "k--", lw=1.3, zorder=5, label=r"$N(0,1)$")
    if CL_EDGE:
        #  Эджворт с ТОЧНОЙ асимметрией шага, для середины набора по g.
        _gm = int(np.median([c[0] for c in GENC]))
        _corr = 1.0 + (GAM_S / (6 * np.sqrt(_gm))) * (zz ** 3 - 3 * zz)
        ax0.plot(zz, _phi(zz) * np.maximum(_corr, 1e-6), "-", color="crimson",
                 lw=1.1, zorder=4,
                 label=r"$+\,\gamma_1/6\sqrt{g}$, $g=%d$" % _gm)
    ax0.set_xlim(*CL_XLIM)
    if CL_LOGY:
        ax0.set_yscale("log"); ax0.set_ylim(1e-5, 1.2)
    ax0.set_xlabel(r"$z=(\ln m-\mu g)/\sigma\sqrt{g}$", labelpad=1)
    ax0.set_ylabel(r"$p(z)$", labelpad=2)
    ax0.legend(frameon=False, loc="lower center", handlelength=1.5, borderpad=0.1)
    ax0.text(0.03, 0.94, "Fragmentation", transform=ax0.transAxes, ha="left",
             va="top", fontsize=7.5, color="0.1", zorder=6,
             bbox=dict(facecolor="white", edgecolor="none", pad=1.2, alpha=0.85))

    # --- низ ---
    ax1 = axc2[1]
    _ab = np.array([c[0] for c in ISOC])
    nB = LogNorm(vmin=_ab.min(), vmax=max(_ab.max(), _ab.min() * 1.0001))
    for age, u, v in ISOC:
        ax1.plot(u, v, "-", lw=0.9, color=CMAP(nB(age)), alpha=0.95, zorder=3)
    ax1.set_xscale("log"); ax1.set_yscale("log")
    ax1.set_xlim(3e-2, 3e1); ax1.set_ylim(1e-4, 2.0)
    ax1.axvline(1.0, color="0.1", lw=0.6, ls=":", zorder=1)
    ax1.set_xlabel(r"$m/\langle m\rangle$", labelpad=1)
    ax1.set_ylabel(r"$m\,dN/dm$  (peak $=1$)", labelpad=2)
    ax1.text(0.03, 0.94, "Coagulation", transform=ax1.transAxes, ha="left",
             va="top", fontsize=7.5, color="0.1", zorder=6,
             bbox=dict(facecolor="white", edgecolor="none", pad=1.2, alpha=0.85))

    figc2.subplots_adjust(left=0.175, right=0.975, bottom=0.105, top=0.985, hspace=0.42)
    if CL_SAVEFIG:
        figc2.savefig(CL_SAVEFIG, bbox_inches="tight"); print("записано ->", CL_SAVEFIG)

try:
    from IPython.display import display
    display(figc2); plt.close(figc2)
except Exception:
    plt.show()

# ---- числа -------------------------------------------------------------
print("ТОЧНЫЕ кумулянты одного дробления, правило %r, w = %.3g:" % (CL_RULE, _W))
print("   mu = %+.5f   sigma^2 = %.5f   gamma1 = %+.4f   -- ИНТЕГРАЛОМ, не фитом"
      % (MU_S, VAR_S, GAM_S))
if _TILT:
    print("   сдвиг на время стоянки: sqrt(g)/b, b = %.3f -- из ЗАКОНА ОЖИДАНИЯ"
          % _B)
    print("   этого прогона, не подогнано под коллапс.")
else:
    print("   поправка на время стоянки ВЫКЛЮЧЕНА (CL_TILT = False)")
print()
print("ВЕРХ, поколения.  frac -- какая доля кривой ВИДНА в коробке (считается,")
print("не подгоняется).  D -- Колмогоров против Phi, перенормированной на то же окно.")
print("%4s %7s %9s %9s" % ("g", "бинов", "frac", "D"))
for g, nb_, frac, D in CREP:
    print("%4d %7d %9.4f %9.4f" % (g, nb_, frac, D))
print("  медиана D = %.4f;  вместе поколения покрывают z от %.2f до %.2f"
      % (np.median([c[3] for c in CREP]),
         min(c[1].min() for c in GENC), max(c[1].max() for c in GENC)))
print("  (у g с малым frac D завышена: там видна одна десятая кривой,")
print("   и на неё приходится весь обрез стоком)")
print()
print("НИЗ, изохроны.  std/<m> -- второй момент масштабной функции, чистое число.")
print("%8s %7s %12s %9s" % ("tau", "бинов", "<m>", "std/<m>"))
for age, nb_, mean, cv in IREP:
    print("%8.1f %7d %12.4g %9.3f" % (age, nb_, mean, cv))
_cv = np.array([r[3] for r in IREP])
print("  разброс std/<m>: %.0f%% вокруг %.3f"
      % (100 * (_cv.max() - _cv.min()) / _cv.mean(), _cv.mean()))
